In [4]:
import os
import shutil
import random

In [5]:
# --- CONFIGURATION ---
DATASET_ROOT = "dataset\\foodseg103"  # Path to your FoodSeg103 root directory
TRAIN_RATIO = 0.70   # 70% for training
VAL_RATIO = 0.15     # 15% for validation
TEST_RATIO = 0.15    # 15% for testing (Must sum up to 1.0)
SEED = 42            # For reproducibility

In [6]:
# Paths
imagesets_dir = os.path.join(DATASET_ROOT, "ImageSets")
img_dir = os.path.join(DATASET_ROOT, "Images", "img_dir")
ann_dir = os.path.join(DATASET_ROOT, "Images", "ann_dir")

print("Configuration loaded. Target directories mapped out.")

Configuration loaded. Target directories mapped out.


In [7]:
def read_image_ids(txt_path):
    if not os.path.exists(txt_path):
        return []
    with open(txt_path, "r") as f:
        return [line.strip() for line in f if line.strip()]

def move_files(ids, src_split, dest_split):
    """Moves images and annotations from old split folders to new staging folders."""
    for img_id in ids:
        # Move raw images (.jpg)
        src_img = os.path.join(img_dir, src_split, f"{img_id}.jpg")
        dest_img = os.path.join(img_dir, dest_split, f"{img_id}.jpg")
        if os.path.exists(src_img):
            os.makedirs(os.path.dirname(dest_img), exist_ok=True)
            shutil.move(src_img, dest_img)

        # Move masks (.png)
        src_ann = os.path.join(ann_dir, src_split, f"{img_id}.png")
        dest_ann = os.path.join(ann_dir, dest_split, f"{img_id}.png")
        if os.path.exists(src_ann):
            os.makedirs(os.path.dirname(dest_ann), exist_ok=True)
            shutil.move(src_ann, dest_ann)

def write_image_ids(txt_path, ids):
    with open(txt_path, "w") as f:
        for img_id in ids:
            f.write(f"{img_id}\n")
            
print("Helper functions defined successfully.")


Helper functions defined successfully.


In [15]:
# 1. Gather all original image IDs
random.seed(SEED)
old_train_ids = read_image_ids(os.path.join(imagesets_dir, "train.txt"))
old_test_ids = read_image_ids(os.path.join(imagesets_dir, "test.txt"))

all_ids = old_train_ids + old_test_ids
if not all_ids:
    raise FileNotFoundError("Error: No image IDs found. Please double-check your DATASET_ROOT path.")

print(f"Total images found in dataset: {len(all_ids)}")

# Shuffle the combined list
random.shuffle(all_ids)

# 2. Calculate new split distributions
total = len(all_ids)
num_train = int(total * TRAIN_RATIO)
num_val = int(total * VAL_RATIO)

train_ids = all_ids[:num_train]
val_ids = all_ids[num_train:num_train + num_val]
test_ids = all_ids[num_train + num_val:]

print(f"Calculated New Splits -> Train: {len(train_ids)} | Val: {len(val_ids)} | Test: {len(test_ids)}")

# 3. Create a NEW dataset copy root (does not overwrite original folders)
OUTPUT_ROOT = DATASET_ROOT + "_split_copy"
out_imagesets_dir = os.path.join(OUTPUT_ROOT, "ImageSets")
out_img_dir = os.path.join(OUTPUT_ROOT, "Images", "img_dir")
out_ann_dir = os.path.join(OUTPUT_ROOT, "Images", "ann_dir")

for split in ["train", "val", "test"]:
    os.makedirs(os.path.join(out_img_dir, split), exist_ok=True)
    os.makedirs(os.path.join(out_ann_dir, split), exist_ok=True)
os.makedirs(out_imagesets_dir, exist_ok=True)

def normalize_id(raw_id):
    """Convert IDs like '00001234.jpg' to base stem '00001234'."""
    return os.path.splitext(os.path.basename(raw_id))[0]

def find_source_file(base_dir, base_id, ext):
    """Find whether this id currently lives in source train or test and return the full source path."""
    for src_split in ["train", "test"]:
        candidate = os.path.join(base_dir, src_split, f"{base_id}.{ext}")
        if os.path.exists(candidate):
            return candidate
    return None

def copy_files(ids, dest_split):
    missing_img = 0
    missing_ann = 0
    copied_img = 0
    copied_ann = 0

    for raw_id in ids:
        base_id = normalize_id(raw_id)

        # Copy raw images (.jpg) from whichever source split actually contains the file
        src_img = find_source_file(img_dir, base_id, "jpg")
        dest_img = os.path.join(out_img_dir, dest_split, f"{base_id}.jpg")
        if src_img is not None:
            shutil.copy2(src_img, dest_img)
            copied_img += 1
        else:
            missing_img += 1

        # Copy masks (.png) from whichever source split actually contains the file
        src_ann = find_source_file(ann_dir, base_id, "png")
        dest_ann = os.path.join(out_ann_dir, dest_split, f"{base_id}.png")
        if src_ann is not None:
            shutil.copy2(src_ann, dest_ann)
            copied_ann += 1
        else:
            missing_ann += 1

    return copied_img, copied_ann, missing_img, missing_ann

print("Copying files into new split copy...")
c_train = copy_files(train_ids, "train")
c_val = copy_files(val_ids, "val")
c_test = copy_files(test_ids, "test")

print("Copy summary (copied_img, copied_ann, missing_img, missing_ann):")
print(f"  Train: {c_train}")
print(f"  Val:   {c_val}")
print(f"  Test:  {c_test}")

# 4. Generate the new text files inside OUTPUT_ROOT/ImageSets
write_image_ids(os.path.join(out_imagesets_dir, "train.txt"), train_ids)
write_image_ids(os.path.join(out_imagesets_dir, "val.txt"), val_ids)
write_image_ids(os.path.join(out_imagesets_dir, "test.txt"), test_ids)

print("\nSuccess! New split copy created.")
print(f"Output dataset root: {OUTPUT_ROOT}")
print("Original dataset remains unchanged.")

Total images found in dataset: 7118
Calculated New Splits -> Train: 4982 | Val: 1067 | Test: 1069
Copying files into new split copy...
Copy summary (copied_img, copied_ann, missing_img, missing_ann):
  Train: (4982, 4982, 0, 0)
  Val:   (1067, 1067, 0, 0)
  Test:  (1069, 1069, 0, 0)

Success! New split copy created.
Output dataset root: dataset\foodseg103_split_copy
Original dataset remains unchanged.
